# Lab 4 Dashboard
Rola Mansour Alghamdi
rolamanss@gmail.com

---
## Step 1: Library Imports
Description: Imports all necessary libraries for data processing (pandas, numpy) and interactive multi-chart dashboard generation (plotly).


In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

---

# Step 2: Data Loading and 12-Month Preprocessing
Description: Loads the dataset, parses date values, and isolates the full 12-month operational data window.



In [4]:
# Read Dataset and parse date field
df = pd.read_csv("logistics_delivery.csv")
df["date"] = pd.to_datetime(df["date"])

# Filter dataset for the full 12-month operational period
last_date = df["date"].max()
full_year = df[df["date"] >= last_date - pd.Timedelta(days=364)]


---
## Step 3: Annual Key Performance Indicators (KPIs) Calculation
Description: Computes 12-month weighted metrics for On-Time Delivery Rate, Total Volume, Average Delivery Duration, and Customer Rating.

In [5]:
# Calculate high-level performance metrics for the full 12 months
avg_on_time = round(
    (full_year["on_time_pct"] * full_year["deliveries_count"]).sum()
    / full_year["deliveries_count"].sum(),
    1,
)
total_deliveries = int(full_year["deliveries_count"].sum())
avg_delivery_time = round(full_year["avg_delivery_time_min"].mean(), 1)
avg_rating = round(full_year["customer_rating"].mean(), 2)

---
#### Step 4: Subplot Grid & Visualizations (Charts 1 to 5)
## Step 4.1: Subplot Grid Structure Setup
Description: Sets up a 4-row layout accommodating 4 top KPI cards (Row 1) and 5 annual analytical charts (Rows 2–4).


In [6]:
fig = make_subplots(
    rows=4,
    cols=4,
    specs=[
        [
            {"type": "domain"},
            {"type": "domain"},
            {"type": "domain"},
            {"type": "domain"},
        ],
        [{"type": "xy", "colspan": 2}, None, {"type": "xy", "colspan": 2}, None],
        [{"type": "xy", "colspan": 2}, None, {"type": "xy", "colspan": 2}, None],
        [{"type": "xy", "colspan": 4}, None, None, None],
    ],
    row_heights=[0.10, 0.30, 0.30, 0.30],
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
    subplot_titles=(
        None,
        None,
        None,
        None,
        "1. ANNUAL HUB RISK STATUS",
        "2. MONTHLY ON-TIME DELIVERY TREND (12 MONTHS)",
        "3. FLEET DURATION BY VEHICLE TYPE",
        "4. VOLUME & CUSTOMER SATISFACTION BY HUB",
        "5. AVERAGE FUEL EXPENDITURE BY VEHICLE TYPE",
    ),
)



## Step 4.2: Chart 1 — Annual Hub Risk Status (Row 2, Column 1)
Description: Displays horizontal bars colored by performance thresholds (Green/Orange/Red) with an 85% target benchmark line.


In [7]:
# Aggregate hub performance for the full 12 months
hub_risk = (
    full_year.groupby("hub", as_index=False)["on_time_pct"]
    .mean()
    .sort_values("on_time_pct", ascending=True)
)


def get_risk_color(val):
    if val >= 90.0:
        return "#00A859"  # Green
    elif val >= 87.0:
        return "#E68A00"  # Orange
    else:
        return "#C8102E"  # Red


hub_risk["color"] = hub_risk["on_time_pct"].apply(get_risk_color)

fig.add_trace(
    go.Bar(
        x=hub_risk["on_time_pct"],
        y=hub_risk["hub"],
        orientation="h",
        marker_color=hub_risk["color"],
        text=hub_risk["on_time_pct"].round(1),
        texttemplate="%{text:.1f}%",
        textposition="inside",
        insidetextanchor="end",
        textfont=dict(color="white", size=11),
        hovertemplate="Hub: %{y}<br>Annual On-Time Rate: %{x:.1f}%<extra></extra>",
        showlegend=False,
    ),
    row=2,
    col=1,
)

# Vertical Target Line at 85% scoped strictly to Chart 1 axes
fig.add_shape(
    type="line",
    x0=85,
    x1=85,
    y0=-0.5,
    y1=len(hub_risk) - 0.5,
    xref="x1",
    yref="y1",
    line=dict(color="#C8102E", width=2, dash="dash"),
)

##Step 4.3: Chart 2 — Monthly On-Time Delivery Trend (Row 2, Column 3)
Description: Plots monthly aggregated punctuality across the 12 months (Jan–Dec) to highlight operational seasonality.

In [8]:

# Aggregating daily data into clean monthly trend points
full_year_trend = full_year.copy()
full_year_trend["month_name"] = full_year_trend["date"].dt.strftime("%b")
full_year_trend["month_num"] = full_year_trend["date"].dt.month

monthly_trend = (
    full_year_trend.groupby(["month_num", "month_name"], as_index=False)[
        "on_time_pct"
    ]
    .mean()
    .sort_values("month_num")
)

fig.add_trace(
    go.Scatter(
        x=monthly_trend["month_name"],
        y=monthly_trend["on_time_pct"],
        mode="lines+markers",
        line=dict(color="#1D9E75", width=2.5),
        marker=dict(size=6),
        hovertemplate="Month: %{x}<br>On-Time Rate: %{y:.1f}%<extra></extra>",
        showlegend=False,
    ),
    row=2,
    col=3,
)



##Step 4.4: Chart 3 — Fleet Duration by Vehicle Type (Row 3, Column 1)
Description: Compares average delivery duration across hub locations categorized by vehicle fleet type over the year.


In [9]:
# Annual grouped duration chart by vehicle type
fleet_duration = (
    full_year.groupby(["hub", "vehicle_type"], as_index=False)[
        "avg_delivery_time_min"
    ]
    .mean()
)

vehicle_colors = {
    "Motorbike": "#00A859",
    "Van": "#E68A00",
    "Truck": "#3B4CCA",
}

for vehicle in ["Motorbike", "Van", "Truck"]:
    veh_data = fleet_duration[fleet_duration["vehicle_type"] == vehicle]
    fig.add_trace(
        go.Bar(
            x=veh_data["hub"],
            y=veh_data["avg_delivery_time_min"],
            name=vehicle,
            marker_color=vehicle_colors.get(vehicle, "#2B5C8F"),
            hovertemplate="Hub: %{x}<br>Type: "
            + vehicle
            + "<br>Duration: %{y:.1f} min<extra></extra>",
            showlegend=True,
        ),
        row=3,
        col=1,
    )

##Step 4.5: Chart 4 — Volume & Customer Satisfaction by Hub (Row 3, Column 3)
Description: Plots annual accumulated delivery volume per hub node.

In [10]:
# Hub volume bar chart
hub_vol_rating = (
    full_year.groupby("hub", as_index=False)
    .agg(
        total_vol=("deliveries_count", "sum"),
        avg_rating=("customer_rating", "mean"),
    )
    .sort_values("total_vol", ascending=False)
)

fig.add_trace(
    go.Bar(
        x=hub_vol_rating["hub"],
        y=hub_vol_rating["total_vol"],
        name="Deliveries Volume",
        marker_color="#2B5C8F",
        text=hub_vol_rating["total_vol"],
        texttemplate="%{text:,.0f}",
        textposition="outside",
        hovertemplate="Hub: %{x}<br>Total Deliveries: %{y:,.0f}<extra></extra>",
        showlegend=False,
    ),
    row=3,
    col=3,
)

##Step 4.6: Chart 5 — Average Fuel Expenditure by Vehicle Type (Row 4, Column 1)
Description: Full-width horizontal bar chart analyzing 12-month average fuel cost per vehicle category.

In [11]:
# 12-Month average fuel expenditure per vehicle category
fuel_veh = (
    full_year.groupby("vehicle_type", as_index=False)["fuel_cost_sar"]
    .mean()
    .sort_values("fuel_cost_sar", ascending=True)
)

fig.add_trace(
    go.Bar(
        x=fuel_veh["fuel_cost_sar"],
        y=fuel_veh["vehicle_type"],
        orientation="h",
        marker_color="#D55E00",
        text=fuel_veh["fuel_cost_sar"],
        texttemplate="SAR %{text:,.2f}",
        textposition="outside",
        hovertemplate="Vehicle: %{y}<br>Avg Fuel Cost: SAR %{x:,.2f}<extra></extra>",
        showlegend=False,
    ),
    row=4,
    col=1,
)



##Step 5: Executive KPI Cards (Row 1)
Description: Places annual summary indicators into Row 1 cards.


In [12]:
# Populating Row 1 single-value Executive KPI Cards
fig.add_trace(
    go.Indicator(
        mode="number",
        value=avg_on_time,
        number={"suffix": "%", "font": {"size": 28, "color": "#1D9E75"}},
        title={"text": "Annual On-Time Rate", "font": {"size": 11}},
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_deliveries,
        number={
            "valueformat": ",.0f",
            "font": {"size": 28, "color": "#2B5C8F"},
        },
        title={"text": "Annual Total Deliveries", "font": {"size": 11}},
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Indicator(
        mode="number",
        value=avg_delivery_time,
        number={"suffix": " m", "font": {"size": 28, "color": "#E69F00"}},
        title={"text": "Avg Duration", "font": {"size": 11}},
    ),
    row=1,
    col=3,
)
fig.add_trace(
    go.Indicator(
        mode="number",
        value=avg_rating,
        number={"font": {"size": 28, "color": "#D55E00"}},
        title={"text": "Avg Customer Rating", "font": {"size": 11}},
    ),
    row=1,
    col=4,
)


---
## Step 6: Executive Layout & Dynamic Room Formatting
Description: Adjusts graph margins, chart headers, legend alignment, and dynamically scales y/x axis limits based on 12-month data metrics


In [13]:
# Configure Executive Layout & Global Theme Formatting
fig.update_layout(
    title=dict(
        text="<b>Logistics Operations Annual Executive Dashboard</b><br><sup>12-Month Performance & Strategic Control View</sup>",
        font=dict(size=18, family="Arial", color="#1A202C"),
        x=0.5,
        xanchor="center",
    ),
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FAFAFA",
    height=1150,
    width=1180,
    margin=dict(l=60, r=60, t=110, b=40),
    barmode="group",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.01,  # Positioned top-center above the grid area
        xanchor="center",
        x=0.5,
        font=dict(size=11),
    ),
)

# Dynamically set headroom for axis bounds
max_vol = hub_vol_rating["total_vol"].max()
fig.update_yaxes(range=[0, max_vol * 1.15], row=3, col=3)

max_fuel = fuel_veh["fuel_cost_sar"].max()
fig.update_xaxes(range=[0, max_fuel * 1.25], row=4, col=1)

# Format Subplot Header Annotations
for ann in fig.layout.annotations:
    ann.font = dict(size=11, family="Arial", color="#1B365D")
    ann.text = f"<b>{ann.text}</b>"

---
## Step 7: Render and HTML Export
Description: Renders the dashboard interactively and exports the full 12-month view to HTML.



In [14]:
# Save the interactive dashboard using Plotly CDN
fig.write_html("logistics_dashboard.html", include_plotlyjs="cdn")

print("Saved: logistics_dashboard.html")
print("Open this file in any browser — no Python needed to view it.")

# Download the HTML file directly to your computer
from google.colab import files

files.download("logistics_dashboard.html")

Saved: logistics_dashboard.html
Open this file in any browser — no Python needed to view it.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>